# Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API

**Objective:** Build a reusable, production-ready ML pipeline to predict customer churn using the Telco dataset.

**Skills:** ML Pipelines, Hyperparameter Tuning (GridSearchCV), Preprocessing, Joblib Export

## Install Dependencies

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

print("✅ All libraries installed successfully!")

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve
)

print("✅ All libraries imported!")

## Load the Telco Churn Dataset

In [ ]:
# Load Telco Customer Churn dataset from GitHub (IBM sample dataset)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

print("📥 Loading Telco Churn dataset...")
df = pd.read_csv(url)

print(f"Dataset shape : {df.shape}")
print(f"Churn rate    : {df['Churn'].value_counts(normalize=True)['Yes']:.1%}")
print("\nFirst few rows:")
display(df.head())

## Exploratory Data Analysis

In [ ]:
# Check missing values and data types
print("📋 Dataset Info:")
print(f"  Shape   : {df.shape}")
print(f"  Missing : {df.isnull().sum().sum()}")
print(f"\nData types:")
print(df.dtypes.value_counts())

# Churn distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Churn counts
df['Churn'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#4CAF50', '#F44336'], edgecolor='white')
axes[0].set_title('Churn Distribution', fontsize=12)
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Churn', 'Churned'], rotation=0)
axes[0].grid(axis='y', alpha=0.3)

# Monthly Charges by Churn
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1],
           boxprops=dict(color='#2196F3'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Monthly Charges by Churn Status', fontsize=12)
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Monthly Charges ($)')

plt.suptitle('')
plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
# ─── Fix known issues ───────────────────────────────────────────────────
# TotalCharges has spaces instead of NaN in some rows
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)

# Drop customerID (not a feature)
df.drop(columns=['customerID'], inplace=True)

# Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# ─── Identify feature types ──────────────────────────────────────────────
numeric_features  = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [c for c in df.columns
                        if c not in numeric_features + ['Churn']]

X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"Numeric features    : {numeric_features}")
print(f"Categorical features: {len(categorical_features)} columns")
print(f"\nTarget distribution:\n{y.value_counts()}")

## Build the Scikit-learn Pipeline

In [ ]:
# ─── Preprocessing sub-pipelines ────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# ─── Full Pipelines ──────────────────────────────────────────────────────
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

print("✅ Pipelines created!")
print("  Pipeline 1 → Logistic Regression")
print("  Pipeline 2 → Random Forest")

## Train-Test Split & Baseline Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train samples : {X_train.shape[0]}")
print(f"Test samples  : {X_test.shape[0]}")

# Baseline fit
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

for name, pipe in [("Logistic Regression", lr_pipeline), ("Random Forest", rf_pipeline)]:
    pred = pipe.predict(X_test)
    acc  = accuracy_score(y_test, pred)
    f1   = f1_score(y_test, pred)
    auc  = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    print(f"\n{name}: Accuracy={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f}")

## Hyperparameter Tuning with GridSearchCV

In [ ]:
# Tune Random Forest pipeline (best baseline) using GridSearchCV
param_grid = {
    'classifier__n_estimators':  [100, 200],
    'classifier__max_depth':     [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("⚙️ Running GridSearchCV (this may take a few minutes)...")
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\n✅ Best parameters : {grid_search.best_params_}")
print(f"   Best CV F1 score : {grid_search.best_score_:.4f}")

## Final Evaluation

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("📊 Tuned Random Forest – Final Results:")
print(f"  Accuracy  : {acc:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churned"]))

## Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ─── Confusion Matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=["No Churn", "Churned"],
            yticklabels=["No Churn", "Churned"])
axes[0].set_title('Confusion Matrix (Tuned RF)', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ─── ROC Curve ───────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#2196F3', linewidth=2, label=f'RF (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Baseline')
axes[1].set_title('ROC Curve', fontsize=12)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Extract feature names after preprocessing
ohe_cols = (best_model.named_steps['preprocessor']
            .named_transformers_['cat']
            .named_steps['onehot']
            .get_feature_names_out(categorical_features).tolist())
all_features = numeric_features + ohe_cols

importances = best_model.named_steps['classifier'].feature_importances_
feat_df = pd.DataFrame({'feature': all_features, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(9, 5))
sns.barplot(data=feat_df, x='importance', y='feature', palette='Blues_r')
plt.title('Top 15 Feature Importances – Random Forest', fontsize=12)
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Export Pipeline with Joblib

In [ ]:
# Save the complete pipeline (includes preprocessor + tuned model)
joblib.dump(best_model, 'churn_pipeline_rf_tuned.joblib')
print("💾 Full pipeline saved: churn_pipeline_rf_tuned.joblib")

# Reload and verify
loaded = joblib.load('churn_pipeline_rf_tuned.joblib')
check = accuracy_score(y_test, loaded.predict(X_test))
print(f"✅ Pipeline reloaded and verified. Accuracy: {check:.4f}")

## Final Summary & Insights

In [ ]:
print("=" * 65)
print("🎉 TASK 2 COMPLETED – END-TO-END ML PIPELINE")
print("=" * 65)
print(f"  Dataset      : Telco Customer Churn")
print(f"  Best Model   : Random Forest (GridSearchCV Tuned)")
print(f"  Accuracy     : {acc:.4f}")
print(f"  F1-Score     : {f1:.4f}")
print(f"  ROC-AUC      : {auc:.4f}")
print()
print("Key Insights:")
print("  • The Pipeline API cleanly separates preprocessing")
print("    from modelling — no data leakage from test set.")
print("  • ColumnTransformer handles numeric/categorical")
print("    features in one reusable step.")
print("  • GridSearchCV with StratifiedKFold gives reliable")
print("    estimates on an imbalanced target (churn ~26%).")
print("  • 'tenure', 'MonthlyCharges', and contract type are")
print("    the strongest predictors of churn.")
print("  • The exported .joblib file is fully production-ready.")
print("=" * 65)